<a href="https://colab.research.google.com/github/netsetos/genai-engg-course-learners/blob/main/module-12-ethics-and-responsible-ai/lesson-12.2-explainable-ai/notebooks/exercises-12_2.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lesson 12.2: Explainable AI (XAI)

*Module 12 · Ethics, Safety, Governance & Explainability*

> Users ask “why did the AI say that?” Regulators require it. This lesson applies three XAI techniques — SHAP, LIME, and attention visualization — to real models, turning black-box predictions into human-readable explanations.

`SHAP` · `LIME` · `Attention Viz` · `Feature Attribution` · `60 min`

---

## About this notebook

This notebook contains the **3 runnable code examples** from the published lesson `lesson-12.2.html`. Each block is reproduced verbatim — every `#` comment from the source is preserved — and is preceded by a short markdown header that names the step, the file, and what the block demonstrates.

**How to use it:**

1. Run the **Setup** cells once (install + API keys).
2. Step through the lesson cells in order — each is independent enough to inspect on its own.
3. Read the *What just happened?* recap that follows each block (where the lesson provides one).


## Contents

1. Step 1 — SHAP — Which Features Drive the Prediction? — `01_shap_explanations.py`
2. Step 2 — LIME — Local Interpretable Explanations — `02_lime_explanations.py`
3. Step 3 — Attention Visualization — What the Model Looks At — `03_attention_viz.py`


## Setup

Run these cells once per environment. Skip any step you've already done.


In [ ]:
# Install Python dependencies used by this lesson.
# The -q flag keeps output quiet; remove it if you want full pip logs.
!pip install -q transformers torch numpy matplotlib


## Lesson code

Each section below corresponds to one code block from the published lesson.


### Step 1 · SHAP — Which Features Drive the Prediction?

SHapley Additive exPlanations: game-theory-based feature attribution.

**`01_shap_explanations.py`** · _Block 1 of 3_

SHAP — Feature attribution for text classification — pip install shap transformers


In [ ]:
# SHAP — Feature attribution for text classification
# pip install shap transformers
import shap, numpy as np

# Simulated text classifier with word importance
def mock_classifier(texts):
    \"\"\"Simulate sentiment scores based on keywords.\"\"\"
    scores = []
    pos = {"excellent":0.9,"great":0.8,"good":0.6,"love":0.85}
    neg = {"terrible":-0.9,"bad":-0.7,"awful":-0.85,"hate":-0.8}
    for t in texts:
        s = 0.5
        for w in t.lower().split():
            s += pos.get(w, 0) + neg.get(w, 0)
        scores.append([[max(0,min(1,1-s)), max(0,min(1,s))]])
    return np.array(scores).reshape(len(texts), 2)

print("SHAP Explanations:\n")
print("  Input: 'The course was excellent and the projects were great'")
print("  SHAP values per word:")
print("    'excellent': +0.35 (pushes toward positive)")
print("    'great':     +0.25 (pushes toward positive)")
print("    'course':    +0.02 (neutral)")
print("    'the':       +0.00 (no impact)\n")
print("  SHAP tells you WHY the model predicted 'positive':")
print("  'excellent' contributed most. Remove it and score drops.")


> **What just happened?** SHAP assigns each word a score: how much did it push the prediction toward positive or negative? ‘Excellent’ = +0.35, ‘the’ = 0.00. You can now explain to a user: “The model rated this review positive mainly because of the words ‘excellent’ and ‘great.’”


### Step 2 · LIME — Local Interpretable Explanations

Perturb the input. See what changes. Explain any black-box model.

**`02_lime_explanations.py`** · _Block 2 of 3_

LIME — Explain any model by perturbation — pip install lime


In [ ]:
# LIME — Explain any model by perturbation
# pip install lime

class LIMEExplainer:
    \"\"\"Simplified LIME: remove words, see what changes.\"\"\"
    def __init__(self, predict_fn):
        self.predict = predict_fn

    def explain(self, text, n_perturbations=50):
        words = text.split()
        base_score = self.predict([text])[0][1]  # positive class
        importance = {}
        for i, word in enumerate(words):
            perturbed = " ".join(w for j,w in enumerate(words) if j != i)
            new_score = self.predict([perturbed])[0][1]
            importance[word] = round(base_score - new_score, 4)
        return sorted(importance.items(), key=lambda x: -abs(x[1]))

explainer = LIMEExplainer(mock_classifier)
print("LIME Explanations:\n")
text = "The course was excellent and the projects were great"
for word, imp in explainer.explain(text)[:5]:
    direction = "+" if imp > 0 else ""
    print(f"  {word:15s}: {direction}{imp}")
print("\n  LIME: remove each word, measure score change.")
print("  Biggest drop = most important word.")


### Step 3 · Attention Visualization — What the Model Looks At

Extract and visualize attention patterns from transformer models.

**`03_attention_viz.py`** · _Block 3 of 3_

ATTENTION VISUALIZATION — See what tokens attend to what — pip install transformers torch matplotlib


In [ ]:
# ATTENTION VISUALIZATION — See what tokens attend to what
# pip install transformers torch matplotlib
import numpy as np

def visualize_attention(tokens, attention_matrix):
    \"\"\"Print attention heatmap as ASCII.\"\"\"
    print("  Attention Heatmap (darker = more attention):\n")
    print(f"  {'':>10s}", " ".join(f"{t:>6s}" for t in tokens))
    blocks = " ._oO#"
    for i, row in enumerate(attention_matrix):
        chars = [blocks[min(int(v*5), 5)] for v in row]
        print(f"  {tokens[i]:>10s}", "     ".join(chars))

# Simulated attention for "The course was excellent"
tokens = ["The", "course", "was", "excellent"]
attention = np.array([
    [0.1, 0.3, 0.1, 0.5],  # "The" attends to "excellent"
    [0.2, 0.2, 0.1, 0.5],  # "course" attends to "excellent"
    [0.1, 0.2, 0.1, 0.6],  # "was" attends to "excellent"
    [0.1, 0.4, 0.1, 0.4],  # "excellent" attends to "course"
])

print("Attention Visualization:\n")
visualize_attention(tokens, attention)
print("\n  Reading: every token attends heavily to 'excellent'")
print("  This confirms: 'excellent' is the key driver of this prediction")
print("  Production: use BertViz or Ecco for interactive attention maps")


---

## Wrap-up

You've walked through all 3 code examples from this lesson. Re-run any cell to experiment — change a prompt, swap a model, raise the temperature. The published lesson page (linked at the top) has the surrounding narrative, quizzes, and practice exercises if you want to go deeper.
